Connected to myenv (Python 3.12.13)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("Impact of AI.pdf")
data = loader.load()
print(len(data))


/var/folders/kh/vhdps2h12vjgcb28www051cw0000gn/T/ipykernel_4863/2347762127.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


9


In [2]:
data

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-08-06T17:18:24+00:00', 'author': 'Aleesha Singh', 'moddate': '2026-08-06T17:18:25+00:00', 'source': 'Impact of AI.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content="Impact of AI on Entry-Level Jobs in India  \n \nAleesha Singh \n Department of Computer Science, University of Delhi  \n BSc(H) Computer Science - VI Semester \n Research Methodology \n Supervisor: Preeti Sehgal | April 2026 \n \n \nAbstract \nArtificial Intelligence is changing the job market over the world and India is also feeling \nthe effects. Artificial Intelligence is especially affecting entry-level jobs which're often the \nfirst jobs that fresh graduates and young workers get. Many of these jobs involve doing \nthe tasks over and over like entering data helping customers making reports and doing \nbasic coding. Since Artificial Intelligence tools can now do many of these tasks quickly \nan

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)


print("Total number of documents: ",len(docs))

Total number of documents:  15


In [10]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from dotenv import load_dotenv
load_dotenv() 



embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

[-0.023955047, 0.011876456, -0.0033613679, -0.0584139, 0.0015592978]

In [11]:
vectorstore = Chroma.from_documents(documents=docs, embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

In [12]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

retrieved_docs = retriever.invoke("Why entry level jobs are most affected by AI?")


In [13]:
len(retrieved_docs)

2

In [14]:
print(retrieved_docs[1].page_content)

Examples include: 
* Entering data into systems 
* Responding to customer questions 
* Preparing reports 
* Testing simple software code 
* Managing schedules and records 
* Screening job applications 
Because these jobs follow fixed rules Artificial Intelligence can often complete them 
faster. With fewer mistakes. 
 
 
4. Negative Impact of AI on Entry-Level Jobs 
4.1 Reduction in Traditional Hiring 
Many companies may hire fresh graduates for routine jobs because Artificial Intelligence 
can now do some of these tasks. For example chatbots can handle customer service 
and coding assistants can support developers. 
4.2 Fewer First-Time Job Opportunities 
Entry-level jobs are often the starting point of a career. If these jobs disappear fresh 
graduates may struggle to gain experience. This can create a situation where 
employers want experience. Students cannot get experience without a first job. 
4.3 Pressure on Salaries


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",temperature=0.3, max_tokens=500)

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [17]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [18]:
response = rag_chain.invoke({"input": "Why entry level jobs are most affected by AI?"})
print(response["answer"])

Entry-level jobs are most affected because they typically involve repetitive tasks that follow fixed rules, making them easy for AI to automate. Because AI can perform these duties faster and with fewer mistakes, companies are reducing traditional hiring for these roles. Consequently, this limits the first-time job opportunities necessary for graduates to gain initial work experience.
